# Step 3: Scores RBF

# Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


# Move to the Dataset Directory in My Drive

In [ ]:
import os
os.chdir("/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data")
!pwd

/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data


# Scores RBF Code:

In [ ]:
"""
Created on Thu May 30 13:34:40 2024

@author: roqui
"""
import pandas            as pd  # for reading the .csv file and related operations
import numpy             as np  # for working with arrays (multi-dimensional)
import os
from ast                 import literal_eval
from numpy import sqrt

def get_MCC(tn, fp, fn, tp):
    numerator = (tp * tn) - (fp * fn)
    denominator = sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    if denominator == 0:
        return (float(numerator) / 1)
    return (float(numerator) / denominator)
    # return (matthews_corrcoef(y_true=y_true, y_pred=y_pred))

def get_specificity(tn, fp, fn, tp):
    if tn + fp == 0:
        return (float(tn) / 1)
    return (float(tn) / (tn + fp))

def get_f1(tp, fp, fn, tn):
    # Calculate precision and recall
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0

    # Calculate F1 score
    if precision + recall != 0:
        f1_score = 2 * (precision * recall) / (precision + recall)
    else:
        f1_score = 0

    return f1_score

def get_likelihood_ratios(tp, fp, fn, tn):
    # Calculate sensitivity and specificity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0

    # Calculate LR+ and LR-
    #lr_plus = sensitivity / (1 - specificity) if (1 - specificity) != 0 else float('inf')
    lr_minus = (1 - sensitivity) / specificity if specificity != 0 else float('inf')

    return lr_minus

def get_ppv(tp, fp):
    # Calculate Positive Predictive Value
    ppv = tp / (tp + fp) if (tp + fp) != 0 else 0
    return ppv

def get_npv(tn, fn):
    # Calculate Negative Predictive Value
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0
    return npv


# Dictionary to hold the data from each sheet
sheets_data = {}
score_rfb   = {}

# Iterate over each sheet in the Excel file and read it into a DataFrame
#for speed in excel_data.sheet_names:
 #   sheets_data[speed] = pd.read_excel(file_path, sheet_name=speed)

path_step1 = 'ML_Step_1.xlsx'
path_step2 = 'Step2_Results_Filtered.xlsx'

# Read the Excel file
excel_step1 = pd.ExcelFile(path_step1)
excel_step2 = pd.ExcelFile(path_step2)


for sheet_name in excel_step1.sheet_names:

    #Read spredsheet every iteration
    df_step1 = pd.read_excel(excel_step1, sheet_name=sheet_name)
    df_step2 = pd.read_excel(excel_step2, sheet_name=sheet_name)

    # GET AS A LIST THE TOP 2 SFS MODELS THAT GET A BETTER PERFORMANCE
    sigificant_sfs_models = list(df_step2['Feature_idx'].iloc)

    # GET THE TOTAL FEATURES COMBINATION THA WERE USED
    lenght_feat = list(df_step2['#Features'].iloc[:])

    # GET THE SFS_COMBINATION_INDX
    name_models  = list(df_step2.iloc[:,0])

    # GET ALL CONFUSION MATRIXS
    matrix_list = list(df_step2['Confusion_Matrix'].iloc[:])
    counter = 0

    # DF for test results
    df_results = pd.DataFrame(columns=['Name_Model', '#Features', 'C_Value', 'Gamma_Value', 'Feature_idx', 'tol',
                                    'CV_accuracy', 'Test_Accuracy', 'Specificity',  'Sensivity', 'NPV', 'PPV',
                                    'Likelihood_Ratio', 'F1', 'MCC', 'Confusion_Matrix', 'Features'])

    for i in sigificant_sfs_models:

        # GET THE ALL INDX PER SFS MODEL
        features = np.array(literal_eval(df_step1.iloc[:, -1][i]))

        # GET THE FEATURE LIST THAT BELONG TO AECH COMBINATION
        feat_comb = features[:lenght_feat[counter]]

        # GET THE Y_TRUE AND Y_PRED FROM EACH MODEL
        matrix = matrix_list[counter]
        matrix = matrix.replace('\n ', ' ').replace('  ', ',').replace('[ ','[').replace(' [',',[').replace(' ',',')
        matrix = np.array(eval(matrix))
        # Extract the elements of the confusion matrix
        tn, fp, fn, tp = matrix.ravel()

        specificity_val = get_specificity(tn, fp, fn, tp)
        npv_val         = get_npv(tn, fn)
        ppv_val         = get_ppv(tp, fp)
        MCC_val = get_MCC(tn, fp, fn, tp)
        F1_val  = get_f1(tp, fp, fn, tn)
        Lh_val  = get_likelihood_ratios(tp, fp, fn, tn)

        # SAVE TO THE DF RESULT
        df_results = pd.concat([df_results,
                                pd.DataFrame({
                                    'Name_Model':       [name_models[counter]],
                                    '#Features':        [df_step2.iloc[counter,4]],
                                    'C_Value':          [df_step2.iloc[counter,5]],
                                    'Gamma_Value':      [df_step2.iloc[counter,6]],
                                    'CV_split':         [df_step2.iloc[counter,10]],
                                    'Random_State':     [df_step2.iloc[counter,11]],
                                    'Feature_idx':      [i],
                                    'tol':              ['RBF'],
                                    'CV_accuracy':      [df_step2.iloc[counter,1]],
                                    'Test_Accuracy':    [df_step2.iloc[counter,2]],
                                    'Specificity':      [specificity_val],
                                    'Sensivity':        [df_step2.iloc[counter,3]],
                                    'NPV':              [npv_val],
                                    'PPV':              [ppv_val],
                                    'Likelihood_Ratio': [Lh_val],
                                    'F1':               [F1_val],
                                    'MCC':              [MCC_val],
                                    'Confusion_Matrix': [df_step2.iloc[counter,8]],
                                    'Features':         [str(feat_comb)]
                                    })],
                            ignore_index=True)


        counter += 1

    # csv_file_name = f'Scores_RBF_{speed}.xlsx'
    # df_result_file_path = os.path.join(class_dir, csv_file_name)
    # df_results.to_excel(df_result_file_path)
    score_rfb[sheet_name] = df_results

with pd.ExcelWriter('Scores_RBF_older.xlsx') as writer:
    for sheet_name, df_results in score_rfb.items():
        df_results.to_excel(writer, sheet_name=sheet_name, index=False)

<ipython-input-3-574392021>:123: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results,
